In [1]:
import cv2
import numpy as np
import os
import json

input_dir = r"C:\Users\xiaoh\OneDrive\Desktop\Final_morphogenesis\05_shared_data\input"
output_dir = r"C:\Users\xiaoh\OneDrive\Desktop\Final_morphogenesis\00_websites\03_generation"
os.makedirs(output_dir, exist_ok=True)
video_path = os.path.join(input_dir, "ice_crystal_01_resized.mp4")
crystal_id = "ice_crystal_01"

print("setup完成")

setup完成


In [7]:
cap = cv2.VideoCapture(video_path)
brightness = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness.append(np.mean(gray))
cap.release()

brightness = np.array(brightness)
peak = int(np.argmax(brightness))

stable = None
window = 20
for i in range(peak, len(brightness) - window):
    if np.std(brightness[i:i+window]) < 0.3:
        stable = i
        break

print(f"peak: {peak}, stable: {stable}")

start = None
for i in range(10, len(brightness)):
    if brightness[i] - brightness[i-10] > 0.5:
        start = i
        break

print(f"start: {start}, peak: {peak}, stable: {stable}")

peak: 241, stable: 329
start: 141, peak: 241, stable: 329


In [8]:
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, stable)
ret, frame = cap.read()
cap.release()

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8,8))
enhanced = clahe.apply(gray)
blur = cv2.GaussianBlur(enhanced, (3,3), 0)
edges = cv2.Canny(blur, 20, 60)

points = np.where(edges > 0)
y_coords = points[0].tolist()
x_coords = points[1].tolist()

print(f"提取到{len(x_coords)}个边缘点")

提取到100695个边缘点


In [5]:
indices = np.random.choice(len(x_coords), 5000, replace=False)
x_sample = [x_coords[i] for i in indices]
y_sample = [y_coords[i] for i in indices]

print(f"采样后: {len(x_sample)}个点")

采样后: 5000个点


In [9]:
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start)

all_points = []  # 每个点带时间信息
frame_count = 0

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    blur = cv2.GaussianBlur(enhanced, (3,3), 0)
    edges = cv2.Canny(blur, 20, 60)
    
    points = np.where(edges > 0)
    for y, x in zip(points[0], points[1]):
        all_points.append({"x": int(x), "y": int(y), "t": frame_count})
    
    frame_count += 1
    if frame_count > stable - start:
        break

cap.release()
print(f"总共{len(all_points)}个带时间的点")

总共26628602个带时间的点


In [21]:
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start)

all_points = []
frame_count = 0
sample_rate = 400  # 每20个点取1个

while True:
    ret, frame = cap.read()
    if not ret:
        break
    if frame_count % 10 != 0:
        frame_count += 1
        continue
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)
    blur = cv2.GaussianBlur(enhanced, (3,3), 0)
    edges = cv2.Canny(blur, 20, 60)
    
    points = np.where(edges > 0)
    ys, xs = points[0], points[1]
    
    # 每帧随机采样
    if len(xs) > 0:
        idx = np.random.choice(len(xs), max(1, len(xs)//sample_rate), replace=False)
        for i in idx:
            all_points.append({"x": int(xs[i]), "y": int(ys[i]), "t": frame_count})
    
    frame_count += 1
    if frame_count > stable - start:
        break

cap.release()
print(f"采样后{len(all_points)}个点")

采样后6924个点


In [22]:
import json

# 导出点数据
with open(os.path.join(output_dir, "points.json"), "w") as f:
    json.dump(all_points, f)

print(f"导出{len(all_points)}个点到points.json")

导出6924个点到points.json


2d 点云

In [23]:
html_content = """<!DOCTYPE html>
<html>
<head>
    <title>Crystal Growth</title>
    <style>
        body { margin: 0; background: #000; overflow: hidden; }
        canvas { display: block; }
    </style>
</head>
<body>
<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>
<script>
const points = """ + json.dumps(all_points) + """;

const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(75, window.innerWidth/window.innerHeight, 0.1, 10000);
const renderer = new THREE.WebGLRenderer();
renderer.setSize(window.innerWidth, window.innerHeight);
document.body.appendChild(renderer.domElement);

camera.position.z = 1000;

const maxT = Math.max(...points.map(p => p.t));
const imgW = """ + str(int(cap_w := cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_WIDTH))) + """;
const imgH = """ + str(int(cv2.VideoCapture(video_path).get(cv2.CAP_PROP_FRAME_HEIGHT))) + """;

const geometry = new THREE.BufferGeometry();
const positions = new Float32Array(points.length * 3);
const colors = new Float32Array(points.length * 3);

points.forEach((p, i) => {
    positions[i*3] = p.x - imgW/2;
    positions[i*3+1] = -(p.y - imgH/2);
    positions[i*3+2] = 0;
    colors[i*3] = 1;
    colors[i*3+1] = 1;
    colors[i*3+2] = 1;
});

geometry.setAttribute('position', new THREE.BufferAttribute(positions, 3));
geometry.setAttribute('color', new THREE.BufferAttribute(colors, 3));

const material = new THREE.PointsMaterial({ size: 2, vertexColors: true });
const pointCloud = new THREE.Points(geometry, material);
scene.add(pointCloud);

let currentT = 0;
const speed = 1;

function updateVisibility() {
    const visible = new Float32Array(points.length * 3);
    points.forEach((p, i) => {
        if (p.t <= currentT) {
            visible[i*3] = 1;
            visible[i*3+1] = 1;
            visible[i*3+2] = 1;
        } else {
            visible[i*3] = 0;
            visible[i*3+1] = 0;
            visible[i*3+2] = 0;
        }
    });
    geometry.attributes.color.array = visible;
    geometry.attributes.color.needsUpdate = true;
}

function animate() {
    requestAnimationFrame(animate);
    currentT += speed;
    if (currentT > maxT) currentT = 0;
    updateVisibility();
    renderer.render(scene, camera);
}

animate();

window.addEventListener('resize', () => {
    camera.aspect = window.innerWidth/window.innerHeight;
    camera.updateProjectionMatrix();
    renderer.setSize(window.innerWidth, window.innerHeight);
});
</script>
</body>
</html>"""

html_path = os.path.join(output_dir, "crystal_growth.html")
with open(html_path, "w") as f:
    f.write(html_content)

print(f"HTML生成完成: {html_path}")

HTML生成完成: C:\Users\xiaoh\OneDrive\Desktop\Final_morphogenesis\00_websites\03_generation\crystal_growth.html


3d 

In [24]:
# 先提取结晶的平均分叉角度和密度
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, stable)
ret, frame = cap.read()
cap.release()

gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8,8))
enhanced = clahe.apply(gray)
blur = cv2.GaussianBlur(enhanced, (3,3), 0)
edges = cv2.Canny(blur, 20, 60)

# 用Hough变换找主要方向
lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=20, 
                         minLineLength=10, maxLineGap=5)

angles = []
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.arctan2(y2-y1, x2-x1) * 180 / np.pi
        angles.append(angle)

avg_angle = np.mean(np.abs(angles)) if angles else 45
branch_factor = min(5, max(2, len(lines)//50)) if lines is not None else 3

print(f"平均分叉角度: {avg_angle:.1f}度")
print(f"分叉系数: {branch_factor}")

平均分叉角度: 48.9度
分叉系数: 5


In [28]:
# 完整的结晶参数提取
cap = cv2.VideoCapture(video_path)
brightness = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    brightness.append(np.mean(gray))
cap.release()

brightness = np.array(brightness)

# 1. 生长速度（brightness曲线斜率）
growth_rate = np.mean(np.diff(brightness[start:peak]))

# 2. 成核位置（第一个结晶出现在哪里）
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, start)
ret, frame = cap.read()
cap.release()
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8,8))
enhanced = clahe.apply(gray)
blur = cv2.GaussianBlur(enhanced, (3,3), 0)
edges = cv2.Canny(blur, 20, 60)
points = np.where(edges > 0)
if len(points[0]) > 0:
    nucleation_y = np.mean(points[0]) / frame.shape[0]
    nucleation_x = np.mean(points[1]) / frame.shape[1]
else:
    nucleation_x, nucleation_y = 0.5, 0.5

# 3. 最终覆盖面积
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, stable)
ret, frame = cap.read()
cap.release()
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8,8))
enhanced = clahe.apply(gray)
blur = cv2.GaussianBlur(enhanced, (3,3), 0)
edges = cv2.Canny(blur, 20, 60)
coverage = np.sum(edges > 0) / edges.size

# 4. 角度分布（不只是平均，而是分布）
lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=20,
                         minLineLength=10, maxLineGap=5)
angles = []
if lines is not None:
    for line in lines:
        x1, y1, x2, y2 = line[0]
        angle = np.arctan2(y2-y1, x2-x1) * 180 / np.pi
        angles.append(angle)

avg_angle = np.mean(np.abs(angles)) if angles else 45
angle_std = np.std(angles) if angles else 20
branch_factor = min(5, max(2, len(lines)//50)) if lines is not None else 3

# 5. 生长时间长度
growth_duration = stable - start

print(f"生长速度: {growth_rate:.4f}")
print(f"成核位置: x={nucleation_x:.2f}, y={nucleation_y:.2f}")
print(f"最终覆盖面积: {coverage:.4f}")
print(f"平均分叉角度: {avg_angle:.1f}度")
print(f"角度标准差: {angle_std:.1f}度")
print(f"分叉系数: {branch_factor}")
print(f"生长时间: {growth_duration}帧")

生长速度: 0.1951
成核位置: x=0.65, y=0.57
最终覆盖面积: 0.0863
平均分叉角度: 48.9度
角度标准差: 54.9度
分叉系数: 5
生长时间: 188帧


In [34]:
# 导出brightness曲线和时间分布
brightness_data = brightness[start:stable].tolist()

# all_points里的时间分布
t_values = [p["t"] for p in all_points]
t_max = max(t_values)

# 归一化时间
t_normalized = [t/t_max for t in t_values]
t_histogram = np.histogram(t_normalized, bins=50)[0].tolist()

data = {
    "brightness": brightness_data,
    "t_histogram": t_histogram,
    "t_max": t_max,
    "growth_rate": float(growth_rate),
    "nucleation_x": float(nucleation_x),
    "nucleation_y": float(nucleation_y),
    "avg_angle": float(avg_angle),
    "angle_std": float(angle_std),
    "branch_factor": int(branch_factor),
    "coverage": float(coverage),
    "growth_duration": int(growth_duration)
}

with open(os.path.join(output_dir, "crystal_data.json"), "w") as f:
    json.dump(data, f)

print("数据导出完成")

数据导出完成


In [42]:
import urllib.request
three_path = os.path.join(output_dir, "three.min.js")
urllib.request.urlretrieve(
    "https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js",
    three_path
)
print("下载完成")

下载完成


In [44]:
html_3d = f"""<!DOCTYPE html>
<html>
<head>
<title>Morphogenesis</title>
<style>
* {{ margin:0; padding:0; box-sizing:border-box; }}
body {{ background:#000; overflow:hidden; font-family:'Courier New',monospace; }}
#controls {{
    position:fixed; bottom:20px; right:20px;
    background:rgba(5,5,10,0.92); border:1px solid rgba(200,200,220,0.15);
    padding:18px; width:230px; opacity:0; pointer-events:none;
    transition:opacity 0.4s; backdrop-filter:blur(12px);
}}
#tz {{ position:fixed; bottom:0; right:0; width:100px; height:100px; z-index:100; }}
#tz:hover + #controls, #controls:hover {{ opacity:1; pointer-events:all; }}
.lb {{ color:rgba(200,210,230,0.5); font-size:9px; letter-spacing:2px; text-transform:uppercase; margin:9px 0 3px; display:flex; justify-content:space-between; }}
.lb span {{ color:rgba(220,230,255,0.85); }}
input[type=range] {{ width:100%; height:1px; -webkit-appearance:none; background:rgba(200,210,255,0.15); outline:none; margin:3px 0; }}
input[type=range]::-webkit-slider-thumb {{ -webkit-appearance:none; width:7px; height:7px; border-radius:50%; background:rgba(200,220,255,0.8); cursor:pointer; }}
.ttl {{ color:rgba(200,210,230,0.35); font-size:8px; letter-spacing:4px; border-bottom:1px solid rgba(200,210,255,0.1); padding-bottom:7px; margin-bottom:2px; }}
button {{ width:100%; margin-top:12px; background:rgba(200,210,255,0.06); border:1px solid rgba(200,210,255,0.2); color:rgba(200,220,255,0.7); padding:7px; cursor:pointer; font-family:inherit; font-size:8px; letter-spacing:3px; text-transform:uppercase; }}
button:hover {{ background:rgba(200,210,255,0.15); }}
</style>
</head>
<body>
<div id="tz"></div>
<div id="controls">
    <div class="ttl">MORPHOGENESIS</div>
    <div class="lb">Growth Speed <span id="v-spd">1.0</span></div>
    <input type="range" id="c-spd" min="0.1" max="4" value="1.0" step="0.1">
    <div class="lb">Branch Depth <span id="v-dep">5</span></div>
    <input type="range" id="c-dep" min="2" max="8" value="5" step="1">
    <div class="lb">Branch Count <span id="v-brc">3</span></div>
    <input type="range" id="c-brc" min="1" max="6" value="3" step="1">
    <div class="lb">Spread Angle <span id="v-ang">49</span>°</div>
    <input type="range" id="c-ang" min="10" max="120" value="49" step="1">
    <div class="lb">Organic Noise <span id="v-noi">0.5</span></div>
    <input type="range" id="c-noi" min="0" max="2" value="0.5" step="0.05">
    <div class="lb">Thickness <span id="v-thk">1.0</span></div>
    <input type="range" id="c-thk" min="0.2" max="4" value="1.0" step="0.1">
    <div class="lb">Transparency <span id="v-trn">0.6</span></div>
    <input type="range" id="c-trn" min="0.1" max="1.0" value="0.6" step="0.05">
    <div class="lb">Roughness <span id="v-rgh">0.4</span></div>
    <input type="range" id="c-rgh" min="0" max="1" value="0.4" step="0.05">
    <div class="lb">Emission <span id="v-emi">0.2</span></div>
    <input type="range" id="c-emi" min="0" max="1" value="0.2" step="0.05">
    <div class="lb">Color Hue <span id="v-hue">210</span>°</div>
    <input type="range" id="c-hue" min="0" max="360" value="210" step="5">
    <div class="lb">Float <span id="v-flt">0.3</span></div>
    <input type="range" id="c-flt" min="0" max="1" value="0.3" step="0.05">
    <button onclick="rebuild()">REGENERATE</button>
</div>

<script src="three.min.js"></script>
<script>
const cd = {json.dumps(data)};
const brightness = cd.brightness;
const minB = Math.min(...brightness), maxB = Math.max(...brightness);

const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(55, innerWidth/innerHeight, 0.1, 5000);
camera.position.set(0, 100, 550);
camera.lookAt(0, 100, 0);

const renderer = new THREE.WebGLRenderer({{antialias:true}});
renderer.setSize(innerWidth, innerHeight);
renderer.setPixelRatio(devicePixelRatio);
renderer.setClearColor(0x000000);
document.body.appendChild(renderer.domElement);

function gp(id) {{ return parseFloat(document.getElementById(id).value); }}
function sr(s) {{ return ((Math.sin(s*127.1+s*311.7)*43758.5453)%1+1)%1; }}

let rootGroup = new THREE.Group();
scene.add(rootGroup);
let activeBranches = [];
let time = 0;

class GrowingBranch {{
    constructor(points, radius, depth, hue, roughness, transparency, emission) {{
        this.points = points;
        this.radius = radius;
        this.depth = depth;
        this.duration = 80 + (6-depth)*25;
        this.progress = 0;
        this.done = false;
        this.childrenSpawned = false;
        this.hue = hue;
        this.roughness = roughness;
        this.transparency = transparency;
        this.emission = emission;
        
        const curve = new THREE.CatmullRomCurve3(points);
        const tubeSeg = Math.max(8, points.length * 3);
        this.geo = new THREE.TubeGeometry(curve, tubeSeg, radius, 8, false);
        this.totalIndex = this.geo.index.count;
        this.geo.setDrawRange(0, 0);
        
        const maxDep = gp('c-dep');
        const t = depth / maxDep;
        const col = new THREE.Color().setHSL(hue/360 + sr(depth)*0.06, 0.3+t*0.2, 0.55+t*0.3);
        this.mat = new THREE.MeshPhysicalMaterial({{
            color: col,
            transparent: true,
            opacity: transparency * (0.3 + t*0.7),
            roughness: roughness,
            metalness: 0.05,
            transmission: 0.5 - roughness*0.3,
            emissive: col,
            emissiveIntensity: emission * t * 0.5,
            side: THREE.DoubleSide
        }});
        
        this.mesh = new THREE.Mesh(this.geo, this.mat);
        rootGroup.add(this.mesh);
    }}
    
    update(dt, speed) {{
        if(this.done) return false;
        this.progress = Math.min(1, this.progress + dt * speed / this.duration);
        this.geo.setDrawRange(0, Math.floor(this.progress * this.totalIndex));
        if(this.progress >= 0.99 && !this.childrenSpawned) {{
            this.childrenSpawned = true;
            this.done = true;
            return true;
        }}
        return false;
    }}
    
    getEndPoint() {{ return this.points[this.points.length-1]; }}
    
    getEndDir() {{
        const n = this.points.length;
        const a = this.points[n-2], b = this.points[n-1];
        return new THREE.Vector3(b.x-a.x, b.y-a.y, b.z-a.z).normalize();
    }}
}}

function makePath(sx,sy,sz, dx,dy,dz, length, noise, seed) {{
    const segs = Math.max(6, 10);
    const pts = [];
    for(let i=0; i<=segs; i++) {{
        const t = i/segs;
        const n = noise * 12 * (1 - t*0.4);
        pts.push(new THREE.Vector3(
            sx + dx*length*t + sr(seed+i*0.3)*n*2-n,
            sy + dy*length*t + sr(seed+i*0.7)*n*2-n - t*t*length*0.15,
            sz + dz*length*t + sr(seed+i*0.5)*n*2-n
        ));
    }}
    return pts;
}}

function spawnBranch(x,y,z, dx,dy,dz, length, depth, seed) {{
    if(depth <= 0 || length < 2) return null;
    const pts = makePath(x,y,z, dx,dy,dz, length, gp('c-noi'), seed);
    const radius = (depth * 0.8 + 0.5) * gp('c-thk') * (0.4 + cd.coverage*2);
    const b = new GrowingBranch(pts, radius, depth, gp('c-hue'), gp('c-rgh'), gp('c-trn'), gp('c-emi'));
    return b;
}}

function spawnChildren(b) {{
    const end = b.getEndPoint();
    const dir = b.getEndDir();
    const numB = Math.max(1, Math.round(gp('c-brc') * (0.6 + sr(b.depth*17)*0.8)));
    const angleRad = gp('c-ang') * Math.PI / 180;
    const newDepth = b.depth - 1;
    const len = b.points[0].distanceTo(b.points[b.points.length-1]);
    const newLen = len * (0.55 + cd.coverage * 1.2 + sr(b.depth)*0.1);
    
    for(let i=0; i<numB; i++) {{
        const spread = (i - numB/2 + 0.5) * angleRad * (0.5 + sr(b.depth*i+3)*0.5);
        const tilt = (sr(b.depth*i+7)-0.5) * angleRad * 0.6;
        const theta = Math.atan2(dir.z, dir.x) + spread;
        const phi = Math.acos(Math.max(-1, Math.min(1, dir.y))) + tilt;
        const nx = Math.sin(phi)*Math.cos(theta);
        const ny = Math.cos(phi);
        const nz = Math.sin(phi)*Math.sin(theta);
        const child = spawnBranch(end.x, end.y, end.z, nx, ny, nz, newLen, newDepth, b.depth*100+i*37);
        if(child) activeBranches.push(child);
    }}
}}

function rebuild() {{
    while(rootGroup.children.length) rootGroup.remove(rootGroup.children[0]);
    activeBranches = [];
    time = 0;
    
    const nx = (cd.nucleation_x-0.5)*80;
    const nz = (cd.nucleation_y-0.5)*80;
    const baseLen = 80 + cd.coverage * 150;
    
    for(let i=0; i<2; i++) {{
        const tilt = (sr(i*17)-0.5)*0.25;
        const tz = (sr(i*31)-0.5)*0.25;
        const b = spawnBranch(
        nx + (sr(i*7)-0.5)*20, -100, nz + (sr(i*13)-0.5)*20,
        tilt, 1, tz, baseLen, gp('c-dep'), i*77
    );
        if(b) activeBranches.push(b);
    }}
}}

scene.add(new THREE.AmbientLight(0x223344, 4));
const l1 = new THREE.PointLight(0xffffff, 3, 1500);
l1.position.set(200, 400, 300); scene.add(l1);
const l2 = new THREE.PointLight(0xaabbff, 2, 1000);
l2.position.set(-300, 200, -100); scene.add(l2);
const l3 = new THREE.PointLight(0xffffff, 1.5, 800);
l3.position.set(0, -100, 400); scene.add(l3);

rebuild();

const clock = new THREE.Clock();
let ft = 0;

function getBSpeed() {{
    const idx = Math.floor((time/300) * brightness.length) % brightness.length;
    const b = brightness[Math.min(idx, brightness.length-1)] || minB;
    return 0.4 + ((b-minB)/(maxB-minB||1)) * 2;
}}

function animate() {{
    requestAnimationFrame(animate);
    const dt = clock.getDelta() * 60;
    ft += dt * 0.01;
    time += dt;
    
    const spd = gp('c-spd') * getBSpeed();
    
    for(let i = activeBranches.length-1; i>=0; i--) {{
        const done = activeBranches[i].update(dt, spd);
        if(done) {{
            spawnChildren(activeBranches[i]);
            activeBranches.splice(i, 1);
        }}
    }}
    
    const flt = gp('c-flt');
    rootGroup.children.forEach((child, i) => {{
        child.position.y += Math.sin(ft*0.8 + i*0.4) * flt * 0.06;
        child.rotation.z += Math.sin(ft*0.3 + i*0.2) * flt * 0.0003;
    }});
    
    rootGroup.rotation.y += 0.003;
    l1.position.x = Math.cos(ft*0.4)*250;
    l1.position.z = Math.sin(ft*0.4)*250;
    
    renderer.render(scene, camera);
}}
animate();

document.querySelectorAll('input[type=range]').forEach(inp => {{
    inp.addEventListener('input', () => {{
        document.getElementById('v-'+inp.id.replace('c-','')).textContent = inp.value;
    }});
}});

window.addEventListener('resize', () => {{
    camera.aspect = innerWidth/innerHeight;
    camera.updateProjectionMatrix();
    renderer.setSize(innerWidth, innerHeight);
}});
</script>
</body>
</html>"""

html_path = os.path.join(output_dir, "crystal_growth_3d.html")
with open(html_path, "w") as f:
    f.write(html_3d)
print(f"完成: {html_path}")

完成: C:\Users\xiaoh\OneDrive\Desktop\Final_morphogenesis\00_websites\03_generation\crystal_growth_3d.html
